<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="./images/btp-banner.gif" alt="BTP A&C">
</div>

# Delivery Delay Prediction with SAP HANA Machine Learning

This notebook demonstrates how to use **SAP HANA Cloud's Predictive Analysis Library (PAL)** — specifically the **Unified Classification** interface — to build a Random Forest model that predicts whether a purchase order will be delivered late.

PAL runs its algorithms as stored procedures directly inside the SAP HANA Cloud engine. All training, scoring, and feature-importance calculations happen **in-database**: data never leaves HANA, which maximises throughput and minimises data-movement overhead.

| Item | Detail |
|------|--------|
| Training table | `SPURCHASE.PO_HISTORY` (600 records) |
| Prediction table | `SPURCHASE.PO_PREDICTION` (9 record) |
| Target column | `Delivery_Delayed` (last column of the dataset) |
| Algorithm | Unified Classification – Random Forest |

## 🎯 Learning Objectives
- How to connect to SAP HANA Cloud using the `hana_ml` Python client and `python-dotenv`
- How to create lazy HANA DataFrames that reference in-database tables without loading data into Python
- How to train a **Random Forest Classifier** using PAL's `UnifiedClassification` API with a stratified train/validation split
- How to evaluate the model with accuracy scores, a confusion matrix, AUC, and an ROC curve
- How to run predictions and explain them with **Shapley values**
- How to inspect feature importances and generate an interactive Unified Report

## 🚨 Prerequisites

Before running this notebook, ensure you have:
- A `.env` file in the project root (or environment variables exported in your shell) containing:
  - `HANA_VECTOR_HOST` — Your HANA Cloud host URL (e.g., `xxx.hanacloud.ondemand.com`)
  - `HANA_VECTOR_USER` — Your HANA username
  - `HANA_VECTOR_PASS` — Your HANA password

## Step 1: Install and Import Required Libraries

The cell below installs all required packages. Key libraries used in this notebook:

| Library | Purpose |
|---------|----------|
| `hdbcli` | Low-level HANA database driver |
| `hana-ml` | High-level Python client for HANA ML / PAL |
| `python-dotenv` | Loads credentials from a `.env` file |
| `matplotlib` | Plotting the ROC curve |
| `plotly` | Interactive charts used by `UnifiedReport` |
| `folium` / `ipywidgets` | Visualisation helpers used by `hana_ml` reports |

> **Tip:** Run the install cell once, then restart the kernel before proceeding.

In [26]:
%pip install hdbcli --break-system-packages
%pip install hana-ml --break-system-packages
%pip install folium --break-system-packages
%pip install ipywidgets --break-system-packages

%pip install plotly --break-system-packages
%pip install nbformat --break-system-packages
%pip install matplotlib --break-system-packages
%pip install -U python-dotenv

# kernel restart required!!!

In [1]:
# Uncomment the line below if hana_ml is not yet installed
# !pip install hana_ml
import os
import hana_ml
from hana_ml import dataframe as hdf
from hana_ml.algorithms.pal import metrics
from hana_ml.algorithms.pal.unified_classification import UnifiedClassification, json2tab_for_reason_code
from hana_ml.visualizers.unified_report import UnifiedReport
from dotenv import load_dotenv

# Load environment variables
load_dotenv()


print('hana_ml version:', hana_ml.__version__)

## Step 2: Connect to SAP HANA Cloud

Credentials are read from environment variables (loaded from `.env` by `python-dotenv`) so that sensitive values are never hard-coded in the notebook.

`hdf.ConnectionContext` opens a persistent, SSL-encrypted connection to the HANA Cloud instance on port 443. The same `myconn` object is reused for all subsequent operations in this notebook.

In [2]:
# Get HANA credentials
HANA_USER = os.getenv('HANA_VECTOR_USER')
HANA_PASS = os.getenv('HANA_VECTOR_PASS')
HANA_HOST = os.getenv('HANA_VECTOR_HOST')

# Establish connection
myconn = hdf.ConnectionContext(
    address=HANA_HOST,
    port=443,
    user=HANA_USER,
    password=HANA_PASS
)
    
    
print(f"Connected to SAP HANA db version {myconn.hana_version()} \nat {HANA_HOST}:443 as user {HANA_USER}")

## Step 3: Create HANA DataFrames

`myconn.table()` creates a **lazy HANA DataFrame** — a Python object that represents a database table without pulling any rows into memory. Only metadata (column names and types) is transferred to the Python client at this point.

This is one of the core advantages of the `hana_ml` API: all heavy lifting (training, scoring, aggregation) happens inside HANA, and Python only receives the final results.

In [3]:
SCHEMA    = 'SPURCHASE'
TBL_TRAIN = 'PO_HISTORY'
TBL_TEST  = 'PO_PREDICTION'

df_train = myconn.table(TBL_TRAIN, schema=SCHEMA)
df_test  = myconn.table(TBL_TEST,  schema=SCHEMA)

print('Training rows  :', df_train.count())
print('Prediction rows:', df_test.count())
print('Columns        :', df_train.columns)

### Identify the Key Column and Target Column

PAL's `UnifiedClassification.fit()` requires explicit `key`, `label`, and `features` arguments. We infer them dynamically from the column list:
- **Key** — first column (unique row identifier)
- **Label** — last column (`Delivery_Delayed`, the binary target)
- **Features** — all remaining columns

In [4]:
# The target is the LAST column of the training table
label_col = df_train.columns[-1]

# The key is the FIRST column (or whichever column uniquely identifies a row)
key_col   = df_train.columns[0]

# All remaining columns are features
feature_cols = [c for c in df_train.columns if c not in (key_col, label_col)]

print(f'Key column   : {key_col}')
print(f'Label column : {label_col}')
print(f'Feature cols : {feature_cols}')

## Step 4: Explore the Training Data

Before training, we inspect the data to understand its shape, statistical properties, and class balance. All aggregations are pushed down to HANA — `.collect()` is called at the end to pull only the small result set into Python.

Key things to look for:
- **Class balance** — an imbalanced `Delivery_Delayed` distribution may require oversampling or cost-sensitive learning
- **Numeric ranges** — outliers or very different scales across features
- **Null values** — PAL handles most missing-value strategies, but it's good to know upfront

In [5]:
# Preview first 5 rows
df_train.head(5).collect()

In [6]:
# Statistical summary
df_train.describe().collect()

In [ ]:
# Distribution of the target class
df_train.agg([('count', label_col, 'COUNT')], group_by=label_col).collect()

In [ ]:
# Preview the prediction record
df_test.collect()

## Step 5: Train the Model with Unified Classification (Random Forest)

`UnifiedClassification` is PAL's unified entry point for supervised classification. Passing `func='RandomForest'` selects the Random Forest algorithm. Key hyperparameters:

| Parameter | Value | Effect |
|-----------|-------|--------|
| `n_estimators` | 10 | Number of decision trees in the forest |
| `max_depth` | 55 | Maximum tree depth — high value allows the forest to model complex interactions |
| `min_samples_leaf` | 1 | Minimum samples required at a leaf node |
| `split_threshold` | 1e-7 | Minimum impurity gain required to split a node |
| `random_state` | 2 | Seed for reproducibility |

The `partition_method='stratified'` argument instructs PAL to perform an **80/20 stratified split** inside HANA, ensuring the class ratio is preserved in both the training and internal validation partitions. The held-out 20 % is used in the scoring step.

In [ ]:
rdt_params = dict(random_state=2,
                  split_threshold=1e-7,
                  min_samples_leaf=1,
                  n_estimators=10,
                  max_depth=55)

uc_rfc = UnifiedClassification(
    func = 'RandomForest', **rdt_params
)

uc_rfc.fit(
    data        = df_train,
    key         = key_col,
    label       = label_col,
    features    = feature_cols,
    partition_method = 'stratified',   # stratified split for balanced classes
    stratified_column= 'Delivery_Delayed',
    partition_random_state = 42,
    training_percent = 0.8             # 80 % train / 20 % internal validation
)

print('Model training complete.')

### Visualize the Model

`UnifiedClassification` provides a built-in `generate_notebook_iframe_report()` method that renders an interactive HTML report directly in the notebook cell. The report includes decision-tree visualisations, feature-importance charts, and validation metrics — all computed from data stored inside HANA.

In [ ]:
from hana_ml.visualizers.dataset_report import DatasetReportBuilder

uc_rfc.build_report()

uc_rfc.generate_notebook_iframe_report()

## Step 6: Evaluate the Model

We evaluate the model against the **internal validation split** (the 20 % held out during `fit`). This is the right choice here because the `PO_PREDICTION` table contains only 1 record — too few to compute meaningful aggregate metrics.

Beyond simple accuracy we inspect:
- **Confusion matrix** — shows true positives, false positives, false negatives, and true negatives
- **AUC (Area Under the ROC Curve)** — threshold-independent measure of discrimination ability; 1.0 is perfect, 0.5 is random
- **ROC curve** — plots True Positive Rate vs False Positive Rate across all classification thresholds

> **Note:** Because the test table (`PO_PREDICTION`) contains only 1 record, scoring is performed on the internal validation split created during training. If the test table contains the true label, you can call `uc_rfc.score(df_test, ...)` directly.

In [ ]:
# --- Option A: score on the training table's held-out validation split ---
# (uses the 20 % partition created by partition_method='stratified' above)
score_result = uc_rfc.score(
    data  = df_train,
    key   = key_col,
    label = label_col
)

print('Accuracy on validation split:', score_result)

### Confusion Matrix and Metrics

After `fit`, PAL stores detailed statistics in attributes on the model object:
- `uc_rfc.statistics_` — per-class and overall metrics (accuracy, precision, recall, F1, AUC)
- `uc_rfc.confusion_matrix_` — the raw confusion matrix
- `uc_rfc.metrics_` — additional curves (e.g., ROC TPR/FPR points)

These are all HANA DataFrames; `.collect()` is used to pull them into pandas for display.

In [ ]:
# Detailed classification metrics stored in the model
uc_rfc.statistics_.collect()

In [ ]:
# Confusion matrix
uc_rfc.confusion_matrix_.collect()

In [ ]:
dtr_auc=uc_rfc.statistics_.filter("STAT_NAME='AUC'").cast('STAT_VALUE','DOUBLE').collect().at[0, 'STAT_VALUE']
dtr_auc

In [ ]:
uc_rfc.metrics_.collect()

In [ ]:
import matplotlib.pyplot as plt

tpr=uc_rfc.metrics_.filter("NAME='ROC_TPR'").select('Y').collect()
fpr=uc_rfc.metrics_.filter("NAME='ROC_FPR'").select('Y').collect()

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=1, label='ROC curve (area = %0.2f)' % dtr_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver operating characteristic')
plt.legend(loc="lower right")
plt.show()

## Step 7: Predict on New Purchase Order Record

`uc_rfc.predict()` scores the single record in `PO_PREDICTION` and returns a HANA DataFrame containing the predicted class label and the associated confidence scores.

We then use **Shapley values** (via `TreeModelDebriefing.shapley_explainer`) to explain *why* the model made its prediction. The summary plot shows how much each feature pushed the prediction towards or away from the positive class — making the result interpretable and auditable.

In [ ]:
predict_res = uc_rfc.predict(
    data = df_test,
    key  = key_col,
    features    = feature_cols
)

predict_res.collect()

In [ ]:
from hana_ml.visualizers.model_debriefing import TreeModelDebriefing

In [ ]:
shapley_explainer = TreeModelDebriefing.shapley_explainer(predict_res, df_test, key=key_col, label=label_col)
shapley_explainer.summary_plot()

## Step 8: Feature Importance

After training, PAL stores the **Gini importance** (Mean Decrease in Impurity) for each feature in `uc_rfc.importance_`. Features with higher importance had a larger average reduction in node impurity across all trees.

This information is useful for:
- **Feature selection** — removing low-importance features to simplify the model
- **Business understanding** — identifying which supply-chain factors most strongly predict delays
- **Data quality** — unexpectedly high importance on a proxy variable may indicate data leakage

In [ ]:
# Variable importance (Gini / Mean Decrease in Impurity)
importance = uc_rfc.importance_
importance.sort('IMPORTANCE', desc=True).collect()

## Step 9: Model Report (Unified Report)

`UnifiedReport` generates a comprehensive, self-contained HTML report covering the full model lifecycle: training configuration, validation metrics, confusion matrix, ROC and Precision-Recall curves, and feature importances — all in one interactive view.

This report can be shared with stakeholders who do not have access to the notebook environment.

In [ ]:
UnifiedReport(uc_rfc).build().display()

## Step 10: Close Connection

Always close the HANA connection when work is complete to release the database session and free up connection-pool resources.

In [ ]:
myconn.close()
print('Connection closed.')